In [4]:
# Download spaCy's English model
import spacy.cli
spacy.cli.download("en_core_web_sm")

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import pandas as pd

df = pd.read_csv("train_cleaned.csv")

In [ ]:
import pandas as pd
import numpy as np
from textblob import TextBlob
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import re
from urllib.parse import urlparse
import nltk
from nltk.corpus import stopwords

# Download necessary NLTK data
nltk.download('stopwords')
nltk.download('punkt')

# Load spaCy model for NER
nlp = spacy.load('en_core_web_sm')

# Function to extract sentiment features
def extract_sentiment(text):
    if isinstance(text, str):
        blob = TextBlob(text)
        return blob.sentiment.polarity, blob.sentiment.subjectivity
    return 0, 0

# Function to extract named entities
def extract_entities(text):
    if isinstance(text, str):
        doc = nlp(text[:1000000])  # Limit text length to avoid memory issues
        persons = sum(1 for ent in doc.ents if ent.label_ == "PERSON")
        orgs = sum(1 for ent in doc.ents if ent.label_ == "ORG")
        gpes = sum(1 for ent in doc.ents if ent.label_ == "GPE")  # Countries, cities, states
        dates = sum(1 for ent in doc.ents if ent.label_ == "DATE")
        return persons, orgs, gpes, dates
    return 0, 0, 0, 0

# Function to extract linguistic features
def extract_linguistic_features(text):
    if isinstance(text, str):
        # Count words
        words = re.findall(r'\b\w+\b', text.lower())
        
        # Get stopwords
        stop_words = set(stopwords.words('english'))
        
        # Calculate stopword ratio
        stopword_count = sum(1 for word in words if word in stop_words)
        stopword_ratio = stopword_count / len(words) if words else 0
        
        # Calculate average word length
        avg_word_len = sum(len(word) for word in words) / len(words) if words else 0
        
        return stopword_ratio, avg_word_len
    return 0, 0

# Apply sentiment analysis
print("Extracting sentiment features...")
df[['title_polarity', 'title_subjectivity']] = pd.DataFrame(df['title'].apply(extract_sentiment).tolist())
df[['text_polarity', 'text_subjectivity']] = pd.DataFrame(df['text'].apply(extract_sentiment).tolist())

# Apply named entity recognition
print("Extracting named entities...")
df[['persons', 'organizations', 'locations', 'dates']] = pd.DataFrame(df['text'].apply(extract_entities).tolist())

# Extract linguistic features
print("Extracting linguistic features...")
df[['stopword_ratio', 'avg_word_len']] = pd.DataFrame(df['text'].apply(extract_linguistic_features).tolist())

# Topic modeling
print("Performing topic modeling...")
tfidf_vectorizer = TfidfVectorizer(max_df=0.95, min_df=2, max_features=1000, stop_words='english')
tfidf = tfidf_vectorizer.fit_transform(df['clean_content'].fillna(''))

# Create LDA model with 5 topics
n_topics = 5
lda = LatentDirichletAllocation(n_components=n_topics, random_state=42)
lda.fit(tfidf)

# Transform the documents to get topic distributions
topic_distributions = lda.transform(tfidf)

# Add topic distribution columns
for i in range(n_topics):
    df[f'topic_{i+1}'] = topic_distributions[:, i]

# Calculate text complexity metrics
df['word_count'] = df['text'].apply(lambda x: len(str(x).split()) if isinstance(x, str) else 0)
df['unique_word_ratio'] = df['text'].apply(lambda x: len(set(str(x).split())) / len(str(x).split()) if isinstance(x, str) and len(str(x).split()) > 0 else 0)

# Calculate title-text similarity (cosine similarity between title and text)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer

def calculate_similarity(row):
    if isinstance(row['title'], str) and isinstance(row['text'], str):
        try:
            vectorizer = CountVectorizer().fit_transform([row['title'], row['text']])
            vectors = vectorizer.toarray()
            if vectors.shape[0] == 2:
                return cosine_similarity([vectors[0]], [vectors[1]])[0][0]
        except:
            pass
    return 0

df['title_text_similarity'] = df.apply(calculate_similarity, axis=1)

print("Feature engineering complete!")
print(f"New dataframe shape: {df.shape}")
print("New features added:", [col for col in df.columns if col not in ['title', 'text', 'label', 'title_len', 'title_excl', 'title_caps', 'text_excl', 'text_ques', 'text_caps', 'clean_content']])

df.to_csv("train_cleaned_1.csv",index = False)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\91903\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\91903\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Extracting sentiment features...
Extracting named entities...


In [5]:
import pandas as pd

df = pd.read_csv("train_cleaned_1.csv")

correlations = df.corr(numeric_only=True)['label'].sort_values(ascending=False)
print(correlations)


label                    1.000000
topic_5                  0.321109
topic_2                  0.304622
topic_3                  0.295891
avg_word_len             0.286985
locations                0.277687
dates                    0.193931
organizations            0.087124
unique_word_ratio        0.076616
title_polarity           0.040792
text_caps                0.017987
title_text_similarity    0.017514
topic_4                 -0.018312
text_polarity           -0.030327
word_count              -0.054465
persons                 -0.149603
text_excl               -0.211137
title_excl              -0.226216
title_subjectivity      -0.291552
text_ques               -0.310775
text_subjectivity       -0.339596
stopword_ratio          -0.344468
title_caps              -0.501256
title_len               -0.548277
topic_1                 -0.686584
Name: label, dtype: float64
